# Static related-gene selection

For every configured target dependency gene, this notebook:

1. matches CRISPR Gene Effect and RNA expression by `ModelID`;
2. tests every expression gene with Pearson, Spearman, and Chi-square;
3. applies Benjamini–Hochberg FDR separately to each method;
4. takes Top-K significant genes from each method;
5. stores an exact `target_gene -> related_genes` mapping for the multi-LLM pipeline;
6. stores the complete association table as CSV for audit.

The report describes statistical associations. It does not establish causality.

In [5]:
import json
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. Configuration

Add target genes to `TARGET_GENES`. The exact `Gene (EntrezID)` column name or a simple gene symbol is accepted.

In [ ]:
EXPRESSION_PATH = Path("OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv")
GENE_EFFECT_PATH = Path("CRISPRGeneEffect.csv")    
OUTPUT_DIR = Path("static_gene_selection")

TARGET_GENES = [
    "A1CF (29974)",
    "AAMP (14)",
]

TOP_K = 20
FDR_ALPHA = 0.05
MIN_N = 20
N_ESSENTIALITY_QUANTILES = 6
EXCLUDE_TARGET_FROM_FEATURES = True

## 2. Helpers

In [10]:
def gene_symbol(column_name):
    return re.sub(r"\s+\(\d+\)$", "", str(column_name)).strip()


def build_gene_map(columns):
    mapping = {}
    for column in columns:
        symbol = gene_symbol(column)
        if symbol not in mapping:
            mapping[symbol] = column
    return mapping


def resolve_gene_column(df, gene):
    if gene in df.columns:
        return gene
    return build_gene_map(df.columns)[gene_symbol(gene)]


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)


def frame_records(df):
    return json.loads(df.to_json(orient="records"))

## 3. Load and align data

In [11]:
expression = pd.read_csv(EXPRESSION_PATH, low_memory=False)
gene_effect = pd.read_csv(GENE_EFFECT_PATH, low_memory=False)
gene_effect = gene_effect.rename(columns={"Unnamed: 0": "ModelID"})

default_mask = (
    expression["IsDefaultEntryForModel"]
    .astype(str)
    .str.lower()
    .isin(["yes", "true", "1"])
)
expression = expression.loc[default_mask].drop_duplicates("ModelID")

expression_gene_columns = [
    column
    for column in expression.columns
    if re.search(r"\(\d+\)$", str(column))
]

common_models = sorted(
    set(expression["ModelID"].dropna())
    & set(gene_effect["ModelID"].dropna())
)

expression_matrix = (
    expression.loc[
        expression["ModelID"].isin(common_models),
        ["ModelID", *expression_gene_columns],
    ]
    .set_index("ModelID")
    .sort_index()
)

gene_effect_matrix = (
    gene_effect[gene_effect["ModelID"].isin(common_models)]
    .drop_duplicates("ModelID")
    .set_index("ModelID")
    .sort_index()
)

print("Expression matrix:", expression_matrix.shape)
print("Gene Effect matrix:", gene_effect_matrix.shape)
print("Configured targets:", TARGET_GENES)

Expression matrix: (1140, 19215)
Gene Effect matrix: (1140, 18531)
Configured targets: ['A1CF (29974)', 'AAMP (14)']


## 4. Statistical tests

`chi2` and `chi2_p` are `NaN` when a valid contingency table cannot be built. `chi2_status` records the reason.

In [12]:
def safe_pearson(x, y, min_n=MIN_N):
    mask = x.notna() & y.notna()
    xx = x[mask].astype(float)
    yy = y[mask].astype(float)
    if len(xx) < min_n or xx.nunique() < 2 or yy.nunique() < 2:
        return np.nan, np.nan, len(xx)
    coefficient, p_value = stats.pearsonr(xx, yy)
    return coefficient, p_value, len(xx)


def safe_spearman(x, y, min_n=MIN_N):
    mask = x.notna() & y.notna()
    xx = x[mask].astype(float)
    yy = y[mask].astype(float)
    if len(xx) < min_n or xx.nunique() < 2 or yy.nunique() < 2:
        return np.nan, np.nan, len(xx)
    coefficient, p_value = stats.spearmanr(xx, yy)
    return coefficient, p_value, len(xx)


def safe_chi_square(x, y, n_quantiles=N_ESSENTIALITY_QUANTILES, min_n=MIN_N):
    mask = x.notna() & y.notna()
    xx = x[mask].astype(float)
    yy = y[mask].astype(float)

    if len(xx) < min_n:
        return np.nan, np.nan, len(xx), np.nan, "insufficient_n"
    if xx.nunique() < 2:
        return np.nan, np.nan, len(xx), np.nan, "constant_expression"
    if yy.nunique() < 2:
        return np.nan, np.nan, len(xx), np.nan, "constant_gene_effect"

    median = xx.median()
    expression_bin = pd.Series(
        np.where(xx <= median, "low", "high"),
        index=xx.index,
    )

    try:
        effect_bin = pd.qcut(yy, q=n_quantiles, duplicates="drop")
    except ValueError:
        return np.nan, np.nan, len(xx), median, "gene_effect_qcut_failed"

    if effect_bin.nunique() < 2:
        return np.nan, np.nan, len(xx), median, "single_gene_effect_bin"

    table = pd.crosstab(expression_bin, effect_bin)
    if table.shape[0] < 2:
        return np.nan, np.nan, len(xx), median, "single_expression_bin"
    if table.shape[1] < 2:
        return np.nan, np.nan, len(xx), median, "single_gene_effect_bin"

    statistic, p_value, _, _ = stats.chi2_contingency(table)
    return statistic, p_value, len(xx), median, "ok"


def add_bh_fdr(df, p_column, fdr_column):
    result = df.copy()
    result[fdr_column] = np.nan
    mask = result[p_column].notna()
    result.loc[mask, fdr_column] = multipletests(
        result.loc[mask, p_column].values,
        alpha=FDR_ALPHA,
        method="fdr_bh",
    )[1]
    return result

## 5. Analyze one target gene

In [13]:
def analyze_target_gene(target_gene):
    target_column = resolve_gene_column(gene_effect_matrix, target_gene)
    target_symbol = gene_symbol(target_column)
    y = gene_effect_matrix[target_column].rename("GeneEffect")
    valid_models = y[y.notna()].index.intersection(expression_matrix.index)
    y = y.loc[valid_models]
    target_expression = expression_matrix.loc[valid_models]

    feature_columns = list(target_expression.columns)
    if EXCLUDE_TARGET_FROM_FEATURES:
        feature_columns = [
            column
            for column in feature_columns
            if gene_symbol(column) != target_symbol
        ]

    rows = []
    for column in feature_columns:
        x = pd.to_numeric(target_expression[column], errors="coerce")
        pearson_r, pearson_p, pearson_n = safe_pearson(x, y)
        spearman_rho, spearman_p, spearman_n = safe_spearman(x, y)
        chi2, chi2_p, chi2_n, expression_median, chi2_status = safe_chi_square(x, y)
        rows.append({
            "gene": gene_symbol(column),
            "expression_column": column,
            "n": min(pearson_n, spearman_n, chi2_n),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "chi2": chi2,
            "chi2_p": chi2_p,
            "chi2_status": chi2_status,
            "expression_median": expression_median,
        })

    results = pd.DataFrame(rows)
    results = add_bh_fdr(results, "pearson_p", "pearson_fdr")
    results = add_bh_fdr(results, "spearman_p", "spearman_fdr")
    results = add_bh_fdr(results, "chi2_p", "chi2_fdr")

    results["pearson_significant"] = results["pearson_fdr"] < FDR_ALPHA
    results["spearman_significant"] = results["spearman_fdr"] < FDR_ALPHA
    results["chi2_significant"] = results["chi2_fdr"] < FDR_ALPHA

    pearson_top = (
        results[results["pearson_significant"]]
        .assign(abs_score=lambda data: data["pearson_r"].abs())
        .sort_values(["pearson_fdr", "abs_score"], ascending=[True, False])
        .head(TOP_K)
    )
    spearman_top = (
        results[results["spearman_significant"]]
        .assign(abs_score=lambda data: data["spearman_rho"].abs())
        .sort_values(["spearman_fdr", "abs_score"], ascending=[True, False])
        .head(TOP_K)
    )
    chi2_top = (
        results[results["chi2_significant"]]
        .sort_values(["chi2_fdr", "chi2"], ascending=[True, False])
        .head(TOP_K)
    )

    pearson_genes = set(pearson_top["gene"])
    spearman_genes = set(spearman_top["gene"])
    chi2_genes = set(chi2_top["gene"])
    selected_genes = pearson_genes | spearman_genes | chi2_genes
    selected = results[results["gene"].isin(selected_genes)].copy()
    selected["selected_by_pearson"] = selected["gene"].isin(pearson_genes)
    selected["selected_by_spearman"] = selected["gene"].isin(spearman_genes)
    selected["selected_by_chi2"] = selected["gene"].isin(chi2_genes)
    selected["n_methods"] = selected[
        ["selected_by_pearson", "selected_by_spearman", "selected_by_chi2"]
    ].sum(axis=1)
    selected["best_fdr"] = selected[
        ["pearson_fdr", "spearman_fdr", "chi2_fdr"]
    ].min(axis=1)
    selected = selected.sort_values(
        ["n_methods", "best_fdr"],
        ascending=[False, True],
    ).reset_index(drop=True)

    report = {
        "target_gene": target_column,
        "target_symbol": target_symbol,
        "n_models": len(y),
        "parameters": {
            "top_k": TOP_K,
            "fdr_alpha": FDR_ALPHA,
            "min_n": MIN_N,
            "essentiality_quantiles": N_ESSENTIALITY_QUANTILES,
            "target_excluded_from_features": EXCLUDE_TARGET_FROM_FEATURES,
        },
        "summary": {
            "tested_expression_genes": len(results),
            "pearson_significant": int(results["pearson_significant"].sum()),
            "spearman_significant": int(results["spearman_significant"].sum()),
            "chi2_significant": int(results["chi2_significant"].sum()),
            "selected_related_genes": len(selected),
        },
        "related_genes": selected["expression_column"].tolist(),
    }
    return report, results, selected

## 6. Run all configured target genes

In [14]:
reports = {}
association_tables = {}
selected_tables = {}

for target_gene in TARGET_GENES:
    print(f"Analyzing {target_gene}...")
    report, associations, selected = analyze_target_gene(target_gene)
    reports[report["target_gene"]] = report
    association_tables[report["target_gene"]] = associations
    selected_tables[report["target_gene"]] = selected
    print(report["summary"])

summary = pd.DataFrame([
    {
        "target_gene": target_gene,
        "n_models": report["n_models"],
        **report["summary"],
    }
    for target_gene, report in reports.items()
])
display(summary)

Analyzing A1CF (29974)...
{'tested_expression_genes': 19214, 'pearson_significant': 401, 'spearman_significant': 113, 'chi2_significant': 13, 'selected_related_genes': 42}
Analyzing AAMP (14)...
{'tested_expression_genes': 19214, 'pearson_significant': 2315, 'spearman_significant': 1998, 'chi2_significant': 256, 'selected_related_genes': 33}


,target_gene,n_models,tested_expression_genes,pearson_significant,spearman_significant,chi2_significant,selected_related_genes
0,A1CF (29974),1140,19214,401,113,13,42
1,AAMP (14),1140,19214,2315,1998,256,33


## 7. Save target and related gene names

`target_related_genes.json` is a minimal dictionary. Each key is an exact target-gene column name and each value is an array of exact related-gene column names. Full association tables remain in CSV files for statistical inspection.

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_directory = OUTPUT_DIR / "reports"
report_directory.mkdir(parents=True, exist_ok=True)

combined_report = {
    target_gene: report["related_genes"]
    for target_gene, report in reports.items()
}

combined_path = OUTPUT_DIR / "target_related_genes.json"
combined_path.write_text(
    json.dumps(combined_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

for target_gene, report in reports.items():
    name = safe_name(target_gene)
    (report_directory / f"{name}_related_genes.json").write_text(
        json.dumps({target_gene: combined_report[target_gene]}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    association_tables[target_gene].to_csv(
        OUTPUT_DIR / f"{name}_all_expression_associations.csv",
        index=False,
    )
    selected_tables[target_gene].to_csv(
        OUTPUT_DIR / f"{name}_related_genes_statistics.csv",
        index=False,
    )

summary.to_csv(OUTPUT_DIR / "target_related_genes_summary.csv", index=False)
print("Target/related gene JSON:", combined_path.resolve())

Target/related gene JSON: /Users/nident/Desktop/JOB/Cancer Cell Line Encyclopedia (CCLE)/data/outputs/static_gene_selection/target_related_genes.json


## 8. Inspect the gene-name mapping

In [16]:
first_target = next(iter(combined_report))
print(json.dumps(
    {first_target: combined_report[first_target]},
    ensure_ascii=False,
    indent=2,
))

{
  "A1CF (29974)": [
    "GREP1 (283875)",
    "ZNF165 (7718)",
    "PADI2 (11240)",
    "DOCK10 (55619)",
    "ARHGAP8 (23779)",
    "PLBD1 (79887)",
    "C1orf115 (79762)",
    "MARVELD3 (91862)",
    "LAMP3 (27074)",
    "IK (3550)",
    "SFXN3 (81855)",
    "G6PC1 (2538)",
    "ENTPD2 (954)",
    "CCNP (79935)",
    "TTC39B (158219)",
    "ESPN (83715)",
    "VSTM5 (387804)",
    "LSR (51599)",
    "CYP8B1 (1582)",
    "IFFO1 (25900)",
    "MMP16 (4325)",
    "HES2 (54626)",
    "MRPS15 (64960)",
    "DISC1 (27185)",
    "SCNN1A (6337)",
    "SUFU (51684)",
    "SLC29A2 (3177)",
    "LYPD6B (130576)",
    "KLC3 (147700)",
    "ING4 (51147)",
    "ARMH3 (79591)",
    "ACOT13 (55856)",
    "ASH2L (9070)",
    "PGAM1 (5223)",
    "EIF3A (8661)",
    "SNX1 (6642)",
    "PGAM4 (441531)",
    "GLYCTK (132158)",
    "CD48 (962)",
    "LSM7 (51690)",
    "EIF5A2 (56648)",
    "ADSS1 (122622)"
  ]
}
